1. Data Preparation – Chuẩn bị & Kiểm tra dữ liệu

In [5]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# 1. Đọc dữ liệu gốc (ép kiểu StateHoliday ngay từ đầu)
train = pd.read_csv("../data/raw/train.csv", dtype={"StateHoliday": "string"})
store = pd.read_csv("../data/raw/store.csv")

print(f"Train shape: {train.shape}")
print(f"Store shape: {store.shape}")

# 2. Kiểm tra tính toàn vẹn khóa chính & khóa ngoại trước khi xử lý
assert train.duplicated(subset=["Store", "Date"]).sum() == 0, "Duplicate (Store, Date) found!"
assert len(set(train["Store"]) - set(store["Store"])) == 0, "Unmatched Store IDs found!"
print("Integrity checks passed: 0 duplicate keys, 100% matched stores.")

Train shape: (1017209, 9)
Store shape: (1115, 10)
Integrity checks passed: 0 duplicate keys, 100% matched stores.


2. Data Cleaning – Làm sạch theo quy tắc Data Dictionary

In [6]:
# --- A. LÀM SẠCH BẢNG STORE (DIM_STORE) ---
store_clean = store.copy()

# 1. Chuẩn hóa nhãn mô tả cho Assortment (phục vụ Power BI)
assortment_map = {"a": "Basic", "b": "Extra", "c": "Extended"}
store_clean["AssortmentLabel"] = store_clean["Assortment"].map(assortment_map)

# 2. Xử lý 3 giá trị thiếu ở CompetitionDistance (Store 291, 622, 879) bằng Median
missing_comp = store_clean[store_clean["CompetitionDistance"].isnull()]
print("Missing CompetitionDistance records:")
display(missing_comp[["Store", "StoreType", "Assortment", "CompetitionDistance"]])

print(
    f"CompetitionDistance stats before imputation -> "
    f"Median: {store_clean['CompetitionDistance'].median():,.2f} | "
    f"Mean: {store_clean['CompetitionDistance'].mean():,.2f} | "
    f"Min: {store_clean['CompetitionDistance'].min():,.2f} | "
    f"Max: {store_clean['CompetitionDistance'].max():,.2f}"
)
comp_dist_median = store_clean["CompetitionDistance"].median()
store_clean["Is_CompDistance_Imputed"] = store_clean["CompetitionDistance"].isnull().astype(int)
store_clean["CompetitionDistance"] = store_clean["CompetitionDistance"].fillna(comp_dist_median)

# 3. Kiểm tra & xử lý năm bất thường (1900) ở CompetitionOpenSinceYear
print("Records with CompetitionOpenSinceYear < 1990:")
display(store_clean[store_clean["CompetitionOpenSinceYear"] < 1990])

# Giữ nguyên NULL cho CompetitionOpenSinceMonth/Year (không điền 0 để tránh sai lệch năm)
# Chuyển sang kiểu Int64 (Nullable Integer của pandas) để không bị hiển thị .0
store_clean["CompetitionOpenSinceMonth"] = store_clean["CompetitionOpenSinceMonth"].astype("Int64")
store_clean["CompetitionOpenSinceYear"] = store_clean["CompetitionOpenSinceYear"].astype("Int64")

# 4. Xử lý nhóm cột Promo2 (544 nulls khớp 100% với Promo2 == 0)
assert store_clean[store_clean["Promo2"] == 0][["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]].isnull().all().all()
store_clean["Promo2SinceWeek"] = store_clean["Promo2SinceWeek"].astype("Int64")
store_clean["Promo2SinceYear"] = store_clean["Promo2SinceYear"].astype("Int64")
store_clean["PromoInterval"] = store_clean["PromoInterval"].fillna("None")


# --- B. LÀM SẠCH BẢNG TRAIN (FACT_SALES) ---
train_clean = train.copy()

# 1. Chuẩn hóa định dạng ngày
train_clean["Date"] = pd.to_datetime(train_clean["Date"])

# 2. Chuẩn hóa cột StateHoliday (giữ cột mã gốc và tạo cột nhãn dễ đọc)
train_clean["StateHoliday"] = train_clean["StateHoliday"].str.strip()
holiday_map = {
    "0": "None",
    "a": "Public Holiday",
    "b": "Easter Holiday",
    "c": "Christmas"
}
train_clean["StateHolidayLabel"] = train_clean["StateHoliday"].map(holiday_map)

# 3. Gắn cờ 54 dòng bất thường (Open == 1 nhưng Sales == 0) - KHÔNG XÓA
train_clean["Is_Zero_Sales_Open"] = ((train_clean["Open"] == 1) & (train_clean["Sales"] == 0)).astype(int)
print(f"Flagged Is_Zero_Sales_Open records: {train_clean['Is_Zero_Sales_Open'].sum()}")

Missing CompetitionDistance records:


,Store,StoreType,Assortment,CompetitionDistance
290,291,d,a,NaN
621,622,a,c,NaN
878,879,d,a,NaN


CompetitionDistance stats before imputation -> Median: 2,325.00 | Mean: 5,404.90 | Min: 20.00 | Max: 75,860.00
Records with CompetitionOpenSinceYear < 1990:


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,AssortmentLabel,Is_CompDistance_Imputed
145,146,d,c,8050.0,10.0,1961.0,1,48.0,2012.0,"Jan,Apr,Jul,Oct",Extended,0
814,815,a,a,590.0,1.0,1900.0,1,40.0,2014.0,"Jan,Apr,Jul,Oct",Basic,0


Flagged Is_Zero_Sales_Open records: 54


3. Feature Engineering – Tạo biến thời gian, đối thủ & khuyến mãi

In [8]:
# 1. Tạo cờ Same-Store Cohort (934 cửa hàng đủ 942 ngày) trong bảng Dim_Store
total_unique_dates = train_clean["Date"].nunique()
store_date_counts = train_clean.groupby("Store")["Date"].nunique()
full_history_ids = store_date_counts[store_date_counts == total_unique_dates].index

store_clean["Is_Full_History_Store"] = store_clean["Store"].isin(full_history_ids).astype(int)
print(f"Full history stores (942 days): {store_clean['Is_Full_History_Store'].sum()} / {len(store_clean)}")

# 2. Tạo các biến thời gian (Date Attributes) trên Fact_Sales
train_clean["Year"] = train_clean["Date"].dt.year
train_clean["Month"] = train_clean["Date"].dt.month
train_clean["Day"] = train_clean["Date"].dt.day
train_clean["Quarter"] = train_clean["Date"].dt.quarter
train_clean["WeekOfYear"] = train_clean["Date"].dt.isocalendar().week.astype(int)
train_clean["IsWeekend"] = train_clean["DayOfWeek"].isin([6, 7]).astype(int)

# 3. Kết hợp tạm thời với Dim_Store để tính toán biến tương tác (Đối thủ & Promo2)
df_merged = train_clean.merge(store_clean, on="Store", how="left")

# 4. Biến thời gian mở cửa đối thủ (CompetitionOpenMonths)
# Loại trừ năm 1900 chưa xác minh khỏi phép tính số tháng
valid_competition_date = (
    df_merged["CompetitionOpenSinceYear"].notna()
    & df_merged["CompetitionOpenSinceMonth"].notna()
    & (df_merged["CompetitionOpenSinceYear"] != 1900)
)

comp_months = (
    (df_merged["Year"] - df_merged["CompetitionOpenSinceYear"]) * 12
    + (df_merged["Month"] - df_merged["CompetitionOpenSinceMonth"])
)

df_merged["CompetitionOpenMonths"] = np.where(
    valid_competition_date,
    comp_months.clip(lower=0),
    np.nan
)

# 5. Biến trạng thái kích hoạt thực tế của Promo2 (Is_Promo2_Active)
month_abbr_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec"
}
df_merged["MonthStr"] = df_merged["Month"].map(month_abbr_map)

iso_cal = df_merged["Date"].dt.isocalendar()
iso_year = iso_cal.year.astype(int)
iso_week = iso_cal.week.astype(int)

after_promo2_start = (
    (df_merged["Promo2"] == 1)
    & (
        (iso_year > df_merged["Promo2SinceYear"])
        | ((iso_year == df_merged["Promo2SinceYear"]) & (iso_week >= df_merged["Promo2SinceWeek"]))
    )
)

in_promo2_interval = [
    m in interval.split(",") if interval != "None" else False
    for m, interval in zip(df_merged["MonthStr"], df_merged["PromoInterval"])
]

df_merged["Is_Promo2_Active"] = (after_promo2_start & np.array(in_promo2_interval)).astype(int)

print("\n=== Promo2 Active Validation ===")
print(df_merged.groupby("Promo2")["Is_Promo2_Active"].agg(["count", "sum", "mean"]))
assert df_merged.loc[df_merged["Promo2"] == 0, "Is_Promo2_Active"].sum() == 0
display
(
    df_merged.loc[
        df_merged["Is_Promo2_Active"] == 1,
        ["Store", "Date", "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval", "Is_Promo2_Active"]
    ].head(10)
)
# 6. Biến hiệu suất giao dịch: Average Spend per Customer
df_merged["SalesPerCustomer"] = np.where(
    df_merged["Customers"] > 0,
    (df_merged["Sales"] / df_merged["Customers"]).round(2),
    np.nan
)

# Đưa các cột engineered thuộc cấp giao dịch về lại bảng Fact_Sales
fact_cols = [
    "Store", "Date", "DayOfWeek", "Year", "Month", "Day", "Quarter", "WeekOfYear", "IsWeekend",
    "Sales", "Customers", "SalesPerCustomer",
    "Open", "Is_Zero_Sales_Open",
    "Promo", "Is_Promo2_Active",
    "StateHoliday", "StateHolidayLabel", "SchoolHoliday",
    "CompetitionOpenMonths"
]
fact_sales_final = df_merged[fact_cols].copy()

Full history stores (942 days): 934 / 1115

=== Promo2 Active Validation ===
         count     sum      mean
Promo2                          
0       508031       0  0.000000
1       509178  152052  0.298622


4. Data Validation & Export – Kiểm tra sau xử lý & Xuất file sạch

In [9]:
print("=== FINAL DATA VALIDATION ===")

# 1. Kiểm tra kích thước & trùng lặp khóa chính
assert len(fact_sales_final) == len(train), "Row count mismatch in Fact_Sales!"
assert fact_sales_final.duplicated(subset=["Store", "Date"]).sum() == 0, "Duplicate PK in Fact_Sales!"
assert len(store_clean) == 1115, "Row count mismatch in Dim_Store!"
assert store_clean.duplicated(subset=["Store"]).sum() == 0, "Duplicate PK in Dim_Store!"

# 2. Kiểm tra tính hợp lệ logic (Sanity Checks)
# <--- ĐÃ CHỈNH SỬA (Mục 4): Bổ sung kiểm tra cohort 934, cờ 54 dòng, Promo2 và thống kê biến mới
assert store_clean["Is_Full_History_Store"].sum() == 934, "Full history store count != 934!"
assert fact_sales_final["Is_Zero_Sales_Open"].sum() == 54, "Zero sales open count != 54!"
assert (df_merged.loc[df_merged["Promo2"] == 0, "Is_Promo2_Active"].sum() == 0), "Promo2=0 has active flags!"
assert (fact_sales_final["Sales"] >= 0).all(), "Negative Sales detected!"
assert (fact_sales_final["Customers"] >= 0).all(), "Negative Customers detected!"

print("Records with Customers = 0 (SalesPerCustomer is NULL):", (fact_sales_final["Customers"] == 0).sum())

print("\n[Fact_Sales Missing Values (Expected in SalesPerCustomer & CompetitionOpenMonths)]")
print(fact_sales_final.isnull().sum()[fact_sales_final.isnull().sum() > 0])

print("\n[Dim_Store Missing Values (Expected in Competition & Promo2 dates)]")
print(store_clean.isnull().sum()[store_clean.isnull().sum() > 0])

print("\n[Summary Statistics of New Engineered Features]")
display(
    fact_sales_final[
        ["Year", "Month", "Quarter", "WeekOfYear", "IsWeekend", "SalesPerCustomer", "CompetitionOpenMonths", "Is_Promo2_Active"]
    ].describe().T
)
# <--- HẾT PHẦN SỬA MỤC 4

print("\nAll validation checks passed successfully!")

# 3. Lưu dữ liệu sạch phục vụ MySQL (Bước 4) & Data Modeling (Bước 5)
import os
os.makedirs("../data/processed", exist_ok=True)

fact_sales_final.to_csv("../data/processed/fact_sales_cleaned.csv", index=False)
store_clean.to_csv("../data/processed/dim_store_cleaned.csv", index=False)

print("Exported:")
print("- ../data/processed/fact_sales_cleaned.csv")
print("- ../data/processed/dim_store_cleaned.csv")

=== FINAL DATA VALIDATION ===
Records with Customers = 0 (SalesPerCustomer is NULL): 172869

[Fact_Sales Missing Values (Expected in SalesPerCustomer & CompetitionOpenMonths)]
SalesPerCustomer         172869
CompetitionOpenMonths    324106
dtype: int64

[Dim_Store Missing Values (Expected in Competition & Promo2 dates)]
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2SinceWeek              544
Promo2SinceYear              544
dtype: int64

[Summary Statistics of New Engineered Features]


,count,mean,std,min,25%,50%,75%,max
Year,1017209.0,2013.832292,0.777396,2013.0,2013.0,2014.00,2014.0,2015.00
Month,1017209.0,5.846762,3.326097,1.0,3.0,6.00,8.0,12.00
Quarter,1017209.0,2.294252,1.081850,1.0,1.0,2.00,3.0,4.00
WeekOfYear,1017209.0,23.615515,14.433381,1.0,11.0,22.00,35.0,52.00
IsWeekend,1017209.0,0.284563,0.451206,0.0,0.0,0.00,1.0,1.00
SalesPerCustomer,844340.0,9.493629,2.197496,0.0,7.9,9.25,10.9,64.96
CompetitionOpenMonths,693103.0,60.200549,56.431209,0.0,14.0,51.00,94.0,645.00
Is_Promo2_Active,1017209.0,0.149480,0.356561,0.0,0.0,0.00,0.0,1.00



All validation checks passed successfully!
Exported:
- ../data/processed/fact_sales_cleaned.csv
- ../data/processed/dim_store_cleaned.csv
